# Fake News Detection - Model Training & Evaluation
This notebook demonstrates the complete process of building an AI system to identify fake news articles using Machine Learning and Natural Language Processing.

## Step 2: Import Libraries & Collect Dataset

In [ ]:
import os
import pandas as pd
import numpy as np
import re
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

## Step 3: Data Cleaning
Load the Fake and Real datasets, merge them, drop null values, and remove duplicate entries.

In [ ]:
fake = pd.read_csv('../dataset/Fake.csv')
real = pd.read_csv('../dataset/True.csv')

fake['label'] = 0
real['label'] = 1

data = pd.concat([fake, real], ignore_index=True)
data.dropna(subset=['text', 'title'], inplace=True)
data.drop_duplicates(subset=['text'], inplace=True)

print(f"Total unique articles: {len(data)}")
data.head()

## Step 4: Text Preprocessing
Clean the text by removing punctuation, URLs, numbers, stopwords, and applying Lemmatization.

In [ ]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    words = text.split()
    cleaned = [lemmatizer.lemmatize(w) for w in words if w not in stop_words]
    return " ".join(cleaned)

# Combine title and text
data['full_content'] = data['title'] + " " + data['text']
# Preprocess a sample for speed (or full if training offline)
sample_df = data.sample(n=10000, random_state=42).copy()
sample_df['cleaned'] = sample_df['full_content'].apply(clean_text)

## Step 5: Exploratory Data Analysis (EDA)
Visualize real vs fake counts, word lengths, and word clouds of common terms.

In [ ]:
# Real vs Fake distribution
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.countplot(x='label', data=sample_df, palette='Set2')
plt.title('Distribution of Fake vs Real News (Sample)')
plt.xticks([0, 1], ['Fake (0)', 'Real (1)'])

plt.subplot(1, 2, 2)
sample_df['label'].value_counts().plot.pie(autopct='%1.1f%%', colors=['coral', 'lightblue'], startangle=90)
plt.title('Percentage of Fake vs Real')
plt.ylabel('')
plt.show()

# Word Cloud for Fake News
fake_text = " ".join(sample_df[sample_df['label'] == 0]['cleaned'])
wordcloud_fake = WordCloud(width=800, height=400, background_color='black').generate(fake_text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud_fake, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - Fake News')
plt.show()

# Word Cloud for Real News
real_text = " ".join(sample_df[sample_df['label'] == 1]['cleaned'])
wordcloud_real = WordCloud(width=800, height=400, background_color='white').generate(real_text)
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud_real, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud - Real News')
plt.show()

## Step 6 & 7: Feature Extraction & Splitting Dataset

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000)
X = vectorizer.fit_transform(sample_df['cleaned'])
y = sample_df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"X_train shape: {X_train.shape}")

## Step 8 & 9: Model Training and Evaluation
Compare Logistic Regression, Naive Bayes, Decision Trees, and Random Forests.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(class_weight='balanced'),
    "Naive Bayes": MultinomialNB(),
    "Decision Tree": DecisionTreeClassifier(max_depth=15, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f"=== {name} Classification Report ===")
    print(classification_report(y_test, preds))
    print("-"*50)

## Step 10: Save Best Model
Save the best classifier (Logistic Regression) and the TF-IDF Vectorizer.

In [ ]:
best_model = LogisticRegression(class_weight='balanced')
best_model.fit(X_train, y_train)

with open('../backend/model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open('../backend/vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Model and Vectorizer successfully saved to backend/")